# Task 3 — zero-aware spending and fixed blend

This notebook reads the existing `eda_combined_v1` dataset from its JSON card and uses the same development/holdout split as `build_model.ipynb`. It never imports notebook variables. Development folds select a fixed blend weight and a nonzero gate threshold; the original holdout checks those choices. All scores here are MAE in THB because the competition documents do not give the NMAE denominator.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostRegressor, CatBoostClassifier
from lightgbm import LGBMRegressor
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task3.models.spending_models import PrefitBlend, PositiveGate
SEED = set_seed()
TARGET, ID = "spending_30d", "player_id"
DATA_NAME = "eda_combined_v1"
card = json.loads((SRC / "task3" / "data" / "eda" / f"{DATA_NAME}.json").read_text(encoding="utf-8"))
assert card["name"] == DATA_NAME and card["task"] == "task3"
train_path = SRC / card["splits"]["train"]["path"]
train = pd.read_parquet(train_path) if card["format"] == "parquet" else pd.read_csv(train_path)
assert train[ID].is_unique and train[TARGET].notna().all()
features = [col for col in train if col not in (ID, TARGET)]
assert len(features) == 48
nominal = ["primary_game", "platform", "seasonal_spending_pattern", "segment"]
assert set(nominal) <= set(features)
y = train[TARGET].astype(float)
assert (y >= 0).all()
bands = pd.cut(y, bins=[-1, 0, 100, 1000, 10000, 50000, np.inf], labels=False)
dev_idx, valid_idx = train_test_split(np.arange(len(y)), test_size=.20,
    stratify=bands, random_state=SEED)
X_dev, X_valid = train.iloc[dev_idx][features], train.iloc[valid_idx][features]
y_dev, y_valid = y.iloc[dev_idx], y.iloc[valid_idx]
assert y_dev.index.intersection(y_valid.index).empty
print('Rows:', len(X_dev), len(X_valid), 'zero share:', round((y_dev == 0).mean(),4))

Rows: 83200 20800 zero share: 0.4812


In [2]:
def preprocessor():
    numeric = [col for col in features if col not in nominal]
    return ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric),
        ("nominal", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value=-1)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), nominal),
    ], remainder="drop", sparse_threshold=0)

def cat_regressor():
    return Pipeline([
        ("preprocess", preprocessor()),
        ("regressor", CatBoostRegressor(loss_function="MAE", iterations=500,
            depth=6, learning_rate=.05, l2_leaf_reg=3., random_seed=SEED,
            thread_count=4, verbose=False, allow_writing_files=False)),
    ])

def lgb_regressor():
    return Pipeline([
        ("preprocess", preprocessor()),
        ("regressor", LGBMRegressor(objective="regression_l1", n_estimators=400,
            learning_rate=.05, num_leaves=31, min_child_samples=50,
            reg_lambda=1., verbosity=-1, random_state=SEED, n_jobs=4)),
    ])

def occurrence_classifier():
    return Pipeline([
        ("preprocess", preprocessor()),
        ("classifier", CatBoostClassifier(loss_function="Logloss", iterations=350,
            depth=6, learning_rate=.05, l2_leaf_reg=3., random_seed=SEED,
            thread_count=4, verbose=False, allow_writing_files=False)),
    ])

SAVE = SRC / "task3" / "save"
def load_saved(name):
    model_card = json.loads((SAVE / f"{name}.json").read_text(encoding="utf-8"))
    assert model_card["data_name"] == DATA_NAME
    return joblib.load(SRC / model_card["model_path"])

baseline_names = ["t3_cat_combined_v1", "t3_lgb_combined_v1", "t3_cat_combined_positive_only_v1"]
baseline_rows = []
for name in baseline_names:
    model = load_saved(name)
    prediction = np.maximum(model.predict(X_valid), 0)
    baseline_rows.append({"model": name, "holdout_MAE_thb_clipped": mean_absolute_error(y_valid, prediction)})
display(pd.DataFrame(baseline_rows).sort_values("holdout_MAE_thb_clipped"))

,model,holdout_MAE_thb_clipped
0,t3_cat_combined_v1,2927.546115
2,t3_cat_combined_positive_only_v1,2934.384718
1,t3_lgb_combined_v1,2957.771950


## 1. Paired development-fold blend search

Each fold fits preprocessing and both component models on that fold’s training rows only. Predictions are cached by fold so rerunning the notebook does not repeat expensive training. The blend weight is chosen from 11 fixed candidates on development folds, then checked once on the original holdout.

In [3]:
RESULTS = SRC / "task3" / "models" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
cv_X = X_dev.reset_index(drop=True)
cv_y = y_dev.reset_index(drop=True)
cv_bands = bands.iloc[dev_idx].reset_index(drop=True)
splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(cv_X, cv_bands))
blend_key = {"dataset": DATA_NAME, "dataset_created_at": card["created_at"],
             "seed": SEED, "folds": 5, "models": ["CatBoost MAE 500x6", "LightGBM L1 400x31"]}
blend_parts = []
for fold, (fit_pos, check_pos) in enumerate(splits, start=1):
    path = RESULTS / f"round3_blend_fold{fold}_v1.parquet"
    meta = RESULTS / f"round3_blend_fold{fold}_v1.json"
    if path.exists() and meta.exists():
        assert json.loads(meta.read_text(encoding="utf-8")) == {"key": blend_key, "fold": fold}
        part = pd.read_parquet(path)
        assert part.row_pos.tolist() == check_pos.tolist()
        print('Loaded blend fold', fold)
    else:
        assert not (path.exists() or meta.exists()), 'Inspect incomplete fold cache'
        cat = cat_regressor(); lgb = lgb_regressor()
        cat.fit(cv_X.iloc[fit_pos], cv_y.iloc[fit_pos])
        lgb.fit(cv_X.iloc[fit_pos], cv_y.iloc[fit_pos])
        part = pd.DataFrame({"fold": fold, "row_pos": check_pos,
            "true": cv_y.iloc[check_pos].to_numpy(),
            "cat": cat.predict(cv_X.iloc[check_pos]),
            "lgb": lgb.predict(cv_X.iloc[check_pos])})
        part.to_parquet(path, index=False)
        meta.write_text(json.dumps({"key": blend_key, "fold": fold}, indent=2) + "\n", encoding="utf-8")
        print('Trained blend fold', fold)
    blend_parts.append(part)
blend_oof = pd.concat(blend_parts, ignore_index=True)
assert blend_oof.row_pos.is_unique and len(blend_oof) == len(cv_X)
weights = np.linspace(0, 1, 11)
blend_scores = pd.DataFrame([
    {"cat_weight": float(weight), "fold": int(fold),
     "MAE_thb": mean_absolute_error(group.true,
         np.maximum(weight*group.cat + (1-weight)*group.lgb, 0))}
    for weight in weights for fold, group in blend_oof.groupby("fold")
])
blend_summary = blend_scores.groupby("cat_weight", as_index=False).agg(
    mean_MAE_thb=("MAE_thb", "mean"), std_MAE_thb=("MAE_thb", "std"))
display(blend_summary.sort_values("mean_MAE_thb").round(2))
best_weight = float(blend_summary.sort_values("mean_MAE_thb").iloc[0].cat_weight)
print('Selected CatBoost weight:', best_weight)
print('Paired fold scores:')
display(blend_scores.pivot(index='fold',columns='cat_weight',values='MAE_thb').round(2))

Loaded blend fold 1
Loaded blend fold 2
Loaded blend fold 3
Loaded blend fold 4
Loaded blend fold 5


,cat_weight,mean_MAE_thb,std_MAE_thb
4,0.4,2989.46,80.16
3,0.3,2990.56,77.23
5,0.5,2992.28,83.32
2,0.2,2995.48,74.67
6,0.6,2998.82,86.77
1,0.1,3003.92,72.87
7,0.7,3008.87,90.98
0,0.0,3015.85,71.53
8,0.8,3022.38,96.00
9,0.9,3039.07,101.38


Selected CatBoost weight: 0.4
Paired fold scores:


cat_weight,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0
fold,,,,,,,,,,,
1,2944.79,2927.51,2913.36,2902.60,2895.62,2892.93,2894.10,2898.10,2905.10,2915.08,2928.03
2,3066.91,3050.92,3038.05,3028.61,3022.34,3019.12,3019.14,3021.81,3027.64,3035.67,3045.74
3,2931.57,2922.45,2916.88,2913.89,2914.85,2919.72,2928.61,2941.41,2957.52,2977.58,3000.34
4,3079.61,3073.68,3071.76,3073.70,3080.26,3091.19,3106.59,3126.58,3151.02,3178.85,3209.62
5,3056.37,3045.05,3037.35,3033.98,3034.22,3038.44,3045.66,3056.46,3070.64,3088.15,3109.21


In [4]:
name = "t3_cat_lgb_blend_cv5_v2"
model_path, card_path, eval_path = (SAVE / f"{name}.joblib", SAVE / f"{name}.json", SAVE / f"{name}-evaluation.json")
if not (model_path.exists() and card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path, card_path, eval_path)), 'Inspect incomplete artifact'
    model = PrefitBlend(load_saved("t3_cat_combined_v1"),
                        load_saved("t3_lgb_combined_v1"), best_weight)
    pred = model.predict(X_valid)
    save_model(task=3, name=name, model=model, data_name=DATA_NAME,
        card={"idea": "Fixed CatBoost/LightGBM blend of development-fitted saved models",
              "components": ["t3_cat_combined_v1", "t3_lgb_combined_v1"],
              "catboost_weight": best_weight, "selection": "five development folds",
              "seed": SEED, "metric_note": "MAE THB; NMAE denominator unavailable"})
    result = evaluate_model(task=3, name=name, score=mean_absolute_error(y_valid, pred),
        metric_name="MAE_thb", higher_is_better=False,
        card={"split": "original stratified 20% holdout", "seed": SEED,
              "weight_selected_on": "development CV only"})
    print(name, result['score'])
else:
    model = load_saved(name)
    pred = model.predict(X_valid)
    print(name, json.loads(eval_path.read_text(encoding="utf-8"))['score'])
print('Holdout MAE by target group:')
for mask_name, mask in [('zero',y_valid == 0),('positive',y_valid > 0),('50k_plus',y_valid >= 50_000)]:
    print(mask_name, mean_absolute_error(y_valid.loc[mask], pred[mask.to_numpy()]))

t3_cat_lgb_blend_cv5_v2

 2907.1322454831306
Holdout MAE by target group:
zero 1736.842043310523
positive 3992.823155932417
50k_plus 20088.775535629044


## 2. Zero-aware two-stage candidate

The first model estimates whether spending is nonzero. The second estimates spending among positive development rows. A probability gate is selected only from development-fold predictions, then the two components are fitted or loaded on the full development portion and checked on the original holdout.

In [5]:
gate_key = {"dataset": DATA_NAME, "dataset_created_at": card["created_at"],
            "seed": SEED, "folds": 5,
            "classifier": "CatBoost Logloss 350x6", "positive_regressor": "CatBoost MAE 500x6"}
gate_parts = []
for fold, (fit_pos, check_pos) in enumerate(splits, start=1):
    path = RESULTS / f"round3_gate_fold{fold}_v1.parquet"
    meta = RESULTS / f"round3_gate_fold{fold}_v1.json"
    if path.exists() and meta.exists():
        assert json.loads(meta.read_text(encoding="utf-8")) == {"key": gate_key, "fold": fold}
        part = pd.read_parquet(path)
        assert part.row_pos.tolist() == check_pos.tolist()
        print('Loaded gate fold',fold)
    else:
        assert not (path.exists() or meta.exists()), 'Inspect incomplete fold cache'
        X_fit,y_fit = cv_X.iloc[fit_pos],cv_y.iloc[fit_pos]
        X_check,y_check = cv_X.iloc[check_pos],cv_y.iloc[check_pos]
        classifier=occurrence_classifier(); positive=cat_regressor()
        classifier.fit(X_fit, (y_fit > 0).astype(int))
        positive.fit(X_fit.loc[y_fit > 0], y_fit.loc[y_fit > 0])
        part=pd.DataFrame({"fold":fold,"row_pos":check_pos,
            "true":y_check.to_numpy(),
            "probability":classifier.predict_proba(X_check)[:,1],
            "positive_amount":positive.predict(X_check)})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({"key":gate_key,"fold":fold},indent=2)+"\n",encoding="utf-8")
        print('Trained gate fold',fold)
    gate_parts.append(part)
gate_oof=pd.concat(gate_parts,ignore_index=True)
assert gate_oof.row_pos.is_unique and len(gate_oof)==len(cv_X)
thresholds=np.linspace(.2,.8,13)
gate_scores=pd.DataFrame([
    {"threshold":float(threshold),"fold":int(fold),
     "MAE_thb":mean_absolute_error(group.true,
         np.where(group.probability>=threshold,
                  np.maximum(group.positive_amount,0),0))}
    for threshold in thresholds for fold,group in gate_oof.groupby('fold')
])
gate_summary=gate_scores.groupby('threshold',as_index=False).agg(
    mean_MAE_thb=('MAE_thb','mean'),std_MAE_thb=('MAE_thb','std'))
display(gate_summary.sort_values('mean_MAE_thb').round(2))
best_threshold=float(gate_summary.sort_values('mean_MAE_thb').iloc[0].threshold)
print('Selected nonzero probability threshold:',best_threshold)
display(gate_scores.pivot(index='fold',columns='threshold',values='MAE_thb').round(2))

Loaded gate fold 1
Loaded gate fold 2
Loaded gate fold 3
Loaded gate fold 4
Loaded gate fold 5


,threshold,mean_MAE_thb,std_MAE_thb
6,0.50,2903.13,88.76
7,0.55,2905.39,89.07
5,0.45,2907.04,88.20
8,0.60,2913.29,91.08
4,0.40,2922.84,87.99
9,0.65,2926.70,87.66
3,0.35,2935.88,88.47
2,0.30,2937.71,88.24
1,0.25,2945.99,87.89
0,0.20,2976.54,88.21


Selected nonzero probability threshold: 0.5


threshold,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80
fold,,,,,,,,,,,,,
1,2894.27,2862.86,2854.55,2852.56,2839.18,2825.66,2821.72,2827.28,2831.95,2847.76,2899.19,2963.22,3022.34
2,3060.68,3029.61,3022.13,3021.06,3007.45,2993.77,2990.69,2989.14,2997.12,3008.86,3066.19,3122.54,3172.04
3,2871.98,2842.82,2834.03,2831.92,2819.78,2802.58,2797.55,2795.43,2802.34,2820.52,2884.52,2940.52,3002.98
4,3004.73,2974.10,2965.35,2963.61,2951.73,2932.62,2929.07,2934.29,2941.52,2953.88,3013.04,3073.66,3132.84
5,3051.05,3020.56,3012.48,3010.25,2996.04,2980.57,2976.62,2980.82,2993.51,3002.47,3065.56,3122.48,3175.11


In [6]:
name = "t3_positive_gate_cv5_v2"
model_path, card_path, eval_path = (SAVE / f"{name}.joblib", SAVE / f"{name}.json", SAVE / f"{name}-evaluation.json")
if not (model_path.exists() and card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path, card_path, eval_path)), 'Inspect incomplete artifact'
    classifier = occurrence_classifier()
    classifier.fit(X_dev, (y_dev > 0).astype(int))
    model = PositiveGate(classifier, load_saved("t3_cat_combined_positive_only_v1"), best_threshold)
    pred = model.predict(X_valid)
    save_model(task=3, name=name, model=model, data_name=DATA_NAME,
        card={"idea": "Nonzero classifier gates positive-only CatBoost amount model",
              "positive_model": "t3_cat_combined_positive_only_v1", "threshold": best_threshold,
              "selection": "five development folds", "seed": SEED,
              "metric_note": "MAE THB; NMAE denominator unavailable"})
    result = evaluate_model(task=3, name=name, score=mean_absolute_error(y_valid,pred),
        metric_name="MAE_thb", higher_is_better=False,
        card={"split":"original stratified 20% holdout","seed":SEED,
              "threshold_selected_on":"development CV only"})
    print(name,result['score'])
else:
    model=load_saved(name); pred=model.predict(X_valid)
    print(name,json.loads(eval_path.read_text(encoding='utf-8'))['score'])
print('Holdout MAE by target group:')
for mask_name,mask in [('zero',y_valid == 0),('positive',y_valid > 0),('50k_plus',y_valid >= 50_000)]:
    print(mask_name,mean_absolute_error(y_valid.loc[mask],pred[mask.to_numpy()]))

t3_positive_gate_cv5_v2 2857.6355602959447
Holdout MAE by target group:
zero 1847.1882195153169
positive 3795.0385149960457
50k_plus 18959.423429324856


## 3. Compare holdout and development-fold evidence

The holdout was used for earlier Task 3 model comparisons, so treat this ranking as exploratory. Prefer a candidate only when development-fold evidence and holdout behavior agree; do not report these MAE values as official NMAE.

In [7]:
names=baseline_names+["t3_cat_lgb_blend_cv5_v2","t3_positive_gate_cv5_v2"]
rows=[]
for name in names:
    model=load_saved(name)
    prediction=np.maximum(model.predict(X_valid),0)
    rows.append({"model":name,
        "holdout_MAE_thb_clipped":mean_absolute_error(y_valid,prediction),
        "MAE_zero":mean_absolute_error(y_valid.loc[y_valid==0],prediction[(y_valid==0).to_numpy()]),
        "MAE_positive":mean_absolute_error(y_valid.loc[y_valid>0],prediction[(y_valid>0).to_numpy()])})
display(pd.DataFrame(rows).sort_values('holdout_MAE_thb_clipped').round(2))
print('Blend CV best:',blend_summary.sort_values('mean_MAE_thb').iloc[0].to_dict())
print('Gate CV best:',gate_summary.sort_values('mean_MAE_thb').iloc[0].to_dict())

,model,holdout_MAE_thb_clipped,MAE_zero,MAE_positive
4,t3_positive_gate_cv5_v2,2857.64,1847.19,3795.04
3,t3_cat_lgb_blend_cv5_v2,2907.13,1736.84,3992.82
0,t3_cat_combined_v1,2927.55,1742.87,4026.58
2,t3_cat_combined_positive_only_v1,2934.38,2134.79,3676.18
1,t3_lgb_combined_v1,2957.77,1733.09,4093.92


Blend CV best: {'cat_weight': 0.4, 'mean_MAE_thb': 2989.4586258953686, 'std_MAE_thb': 80.15897821184062}
Gate CV best: {'threshold': 0.5, 'mean_MAE_thb': 2903.130663456125, 'std_MAE_thb': 88.76144818386082}


## 4. Can a small blend stabilize the zero-aware model?

The zero-aware model wins, but its zero-target MAE is worse than the CatBoost/LightGBM blend. Use the already saved, paired development-fold predictions to choose one fixed mixture. This is a small follow-up search on the same development folds, so its CV gain is exploratory. Evaluate the chosen mixture on the existing holdout once.

In [8]:
from src.task3.models.spending_models import WeightedPair
paired_gate = gate_oof.sort_values('row_pos').reset_index(drop=True)
paired_blend = blend_oof.sort_values('row_pos').reset_index(drop=True)
assert paired_gate.row_pos.equals(paired_blend.row_pos)
assert paired_gate.true.equals(paired_blend.true)
gate_pred = np.where(paired_gate.probability >= best_threshold,
    np.maximum(paired_gate.positive_amount,0),0)
blend_pred = np.maximum(best_weight*paired_blend.cat+(1-best_weight)*paired_blend.lgb,0)
pair_weights = np.linspace(0,1,11)
pair_scores = pd.DataFrame([
    {'gate_weight':float(weight),'fold':int(fold),
     'MAE_thb':mean_absolute_error(paired_gate.loc[mask,'true'],
         (weight*gate_pred+(1-weight)*blend_pred)[mask.to_numpy()])}
    for weight in pair_weights
    for fold,mask in [(value,paired_gate.fold==value) for value in sorted(paired_gate.fold.unique())]
])
pair_summary=pair_scores.groupby('gate_weight',as_index=False).agg(
    mean_MAE_thb=('MAE_thb','mean'),std_MAE_thb=('MAE_thb','std'))
display(pair_summary.sort_values('mean_MAE_thb').round(2))
best_pair_weight=float(pair_summary.sort_values('mean_MAE_thb').iloc[0].gate_weight)
print('Selected gate weight:',best_pair_weight)
display(pair_scores.pivot(index='fold',columns='gate_weight',values='MAE_thb').round(2))

,gate_weight,mean_MAE_thb,std_MAE_thb
8,0.8,2895.92,83.67
7,0.7,2897.36,81.84
9,0.9,2897.82,86.15
6,0.6,2902.04,80.59
10,1.0,2903.13,88.76
5,0.5,2909.91,79.73
4,0.4,2920.39,79.24
3,0.3,2933.86,79.14
2,0.2,2950.07,79.31
1,0.1,2968.69,79.59


Selected gate weight: 0.8


gate_weight,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0
fold,,,,,,,,,,,
1,2895.62,2876.80,2860.57,2846.53,2835.17,2826.31,2819.63,2816.37,2815.58,2817.01,2821.72
2,3022.34,3007.27,2994.24,2983.57,2975.62,2970.42,2967.84,2969.01,2973.69,2980.85,2990.69
3,2914.85,2890.35,2867.79,2848.44,2832.09,2819.00,2808.76,2801.33,2797.07,2795.71,2797.55
4,3080.26,3051.86,3025.73,3002.00,2981.28,2964.08,2949.32,2937.39,2929.51,2927.24,2929.07
5,3034.22,3017.18,3002.05,2988.76,2977.81,2969.73,2964.64,2962.72,2963.76,2968.29,2976.62


In [9]:
name='t3_gate_blend_cv5_v2'
model_path,card_path,eval_path=(SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json')
if not (model_path.exists() and card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path,card_path,eval_path)), 'Inspect incomplete artifact'
    model=WeightedPair(load_saved('t3_positive_gate_cv5_v2'),
                       load_saved('t3_cat_lgb_blend_cv5_v2'),best_pair_weight)
    pred=model.predict(X_valid)
    save_model(task=3,name=name,model=model,data_name=DATA_NAME,
        card={'idea':'Fixed blend of the selected zero-aware model and CatBoost/LightGBM blend',
              'components':['t3_positive_gate_cv5_v2','t3_cat_lgb_blend_cv5_v2'],
              'gate_weight':best_pair_weight,'selection':'same five development folds',
              'seed':SEED,'metric_note':'MAE THB; NMAE denominator unavailable'})
    result=evaluate_model(task=3,name=name,score=mean_absolute_error(y_valid,pred),
        metric_name='MAE_thb',higher_is_better=False,
        card={'split':'original stratified 20% holdout','seed':SEED,
              'weight_selected_on':'development CV only'})
    print(name,result['score'])
else:
    model=load_saved(name);pred=model.predict(X_valid)
    print(name,json.loads(eval_path.read_text(encoding='utf8'))['score'])
for group,mask in [('zero',y_valid==0),('positive',y_valid>0),('50k_plus',y_valid>=50_000)]:
    print(group,mean_absolute_error(y_valid.loc[mask],pred[mask.to_numpy()]))

t3_gate_blend_cv5_v2 2845.0992169223155
zero 1825.1189842743577
positive 3791.3459387764437
50k_plus 19060.71616308467


## 5. Is the nonzero classifier the bottleneck?

Keep the CatBoost positive-amount model and the same five development folds, but replace only the nonzero classifier with LightGBM. This isolates the occurrence decision. Tune the probability threshold on development-fold predictions, then check the resulting saved model on the existing holdout.

In [10]:
from lightgbm import LGBMClassifier

def lgb_occurrence_classifier():
    return Pipeline([
        ('preprocess',preprocessor()),
        ('classifier',LGBMClassifier(n_estimators=400,learning_rate=.05,
            num_leaves=31,min_child_samples=50,reg_lambda=1.,verbosity=-1,
            random_state=SEED,n_jobs=4)),
    ])

lgb_gate_key={'dataset':DATA_NAME,'dataset_created_at':card['created_at'],
    'seed':SEED,'folds':5,'classifier':'LightGBM 400x31',
    'positive_regressor':'same fold CatBoost positive-only predictions as round3_gate'}
lgb_parts=[]
for fold,(fit_pos,check_pos) in enumerate(splits,start=1):
    path=RESULTS/f'round3_lgb_gate_fold{fold}_v1.parquet'
    meta=RESULTS/f'round3_lgb_gate_fold{fold}_v1.json'
    if path.exists() and meta.exists():
        assert json.loads(meta.read_text(encoding='utf8'))=={'key':lgb_gate_key,'fold':fold}
        part=pd.read_parquet(path)
        assert part.row_pos.tolist()==check_pos.tolist()
        print('Loaded LightGBM gate fold',fold)
    else:
        assert not (path.exists() or meta.exists()),'Inspect incomplete fold cache'
        classifier=lgb_occurrence_classifier()
        classifier.fit(cv_X.iloc[fit_pos],(cv_y.iloc[fit_pos]>0).astype(int))
        part=pd.DataFrame({'fold':fold,'row_pos':check_pos,
            'probability':classifier.predict_proba(cv_X.iloc[check_pos])[:,1]})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({'key':lgb_gate_key,'fold':fold},indent=2)+'\n',encoding='utf8')
        print('Trained LightGBM gate fold',fold)
    lgb_parts.append(part)
lgb_gate_oof=pd.concat(lgb_parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
paired_amount=gate_oof.sort_values('row_pos').reset_index(drop=True)
assert lgb_gate_oof.row_pos.equals(paired_amount.row_pos)
lgb_threshold_scores=pd.DataFrame([
    {'threshold':float(threshold),'fold':int(fold),
     'MAE_thb':mean_absolute_error(paired_amount.loc[mask,'true'],
         np.where(lgb_gate_oof.loc[mask,'probability']>=threshold,
                  np.maximum(paired_amount.loc[mask,'positive_amount'],0),0))}
    for threshold in np.linspace(.2,.8,13)
    for fold,mask in [(value,lgb_gate_oof.fold==value) for value in sorted(lgb_gate_oof.fold.unique())]
])
lgb_threshold_summary=lgb_threshold_scores.groupby('threshold',as_index=False).agg(
    mean_MAE_thb=('MAE_thb','mean'),std_MAE_thb=('MAE_thb','std'))
display(lgb_threshold_summary.sort_values('mean_MAE_thb').round(2))
lgb_best_threshold=float(lgb_threshold_summary.sort_values('mean_MAE_thb').iloc[0].threshold)
print('Selected LightGBM gate threshold:',lgb_best_threshold)
display(lgb_threshold_scores.pivot(index='fold',columns='threshold',values='MAE_thb').round(2))

Loaded LightGBM gate fold 1
Loaded LightGBM gate fold 2
Loaded LightGBM gate fold 3
Loaded LightGBM gate fold 4
Loaded LightGBM gate fold 5


,threshold,mean_MAE_thb,std_MAE_thb
6,0.50,2903.70,89.26
7,0.55,2906.31,89.97
5,0.45,2906.76,88.67
8,0.60,2916.25,91.42
4,0.40,2919.10,87.54
3,0.35,2933.55,88.64
2,0.30,2937.96,88.41
9,0.65,2941.36,91.33
1,0.25,2945.70,88.08
0,0.20,2970.99,88.20


Selected LightGBM gate threshold: 0.5


threshold,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80
fold,,,,,,,,,,,,,
1,2888.65,2862.27,2854.74,2850.05,2835.98,2822.62,2818.75,2820.13,2829.97,2852.98,2899.20,2968.78,3032.27
2,3055.21,3030.00,3022.62,3019.57,3004.95,2993.07,2991.75,2994.18,3002.80,3028.63,3068.39,3121.32,3198.12
3,2866.36,2842.37,2834.14,2829.57,2817.42,2804.06,2799.75,2801.94,2808.44,2836.67,2882.75,2938.33,3016.71
4,2999.80,2974.14,2965.16,2960.85,2944.70,2931.32,2931.63,2934.54,2946.75,2968.03,3011.32,3079.23,3131.80
5,3044.93,3019.71,3013.16,3007.70,2992.44,2982.74,2976.61,2980.77,2993.31,3020.52,3058.69,3108.32,3162.45


In [11]:
name='t3_lgb_occurrence_gate_cv5_v2'
model_path,card_path,eval_path=(SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json')
if not (model_path.exists() and card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path,card_path,eval_path)),'Inspect incomplete artifact'
    classifier=lgb_occurrence_classifier()
    classifier.fit(X_dev,(y_dev>0).astype(int))
    model=PositiveGate(classifier,load_saved('t3_cat_combined_positive_only_v1'),lgb_best_threshold)
    pred=model.predict(X_valid)
    save_model(task=3,name=name,model=model,data_name=DATA_NAME,
        card={'idea':'LightGBM nonzero classifier with CatBoost positive-amount model',
              'positive_model':'t3_cat_combined_positive_only_v1',
              'threshold':lgb_best_threshold,'selection':'five development folds',
              'seed':SEED,'metric_note':'MAE THB; NMAE denominator unavailable'})
    result=evaluate_model(task=3,name=name,score=mean_absolute_error(y_valid,pred),
        metric_name='MAE_thb',higher_is_better=False,
        card={'split':'original stratified 20% holdout','seed':SEED,
              'threshold_selected_on':'development CV only'})
    print(name,result['score'])
else:
    model=load_saved(name);pred=model.predict(X_valid)
    print(name,json.loads(eval_path.read_text(encoding='utf8'))['score'])
for group,mask in [('zero',y_valid==0),('positive',y_valid>0),('50k_plus',y_valid>=50_000)]:
    print(group,mean_absolute_error(y_valid.loc[mask],pred[mask.to_numpy()]))

t3_lgb_occurrence_gate_cv5_v2 2858.1099819721844
zero 1841.804785521525
positive 3800.947332896291
50k_plus 18959.423429324856


## 6. Where does the remaining error come from?

Use paired out-of-fold development predictions for the selected fixed 80/20 candidate. Separate zero, small positive, and large positive targets; report each group’s share of total absolute error. This is an error audit, not a new model selection step.

In [12]:
selected_oof = best_pair_weight*gate_pred + (1-best_pair_weight)*blend_pred
assert len(selected_oof) == len(paired_gate)
audit = pd.DataFrame({'true':paired_gate.true.to_numpy(),
    'prediction':selected_oof,'gate_probability':paired_gate.probability.to_numpy()})
audit['absolute_error'] = np.abs(audit.true-audit.prediction)
audit['band'] = pd.cut(audit.true,
    bins=[-1,0,100,1000,10000,50000,np.inf],
    labels=['zero','(0,100]','(100,1k]','(1k,10k]','(10k,50k]','50k+'])
by_band=audit.groupby('band',observed=True).agg(
    rows=('true','size'),mean_target=('true','mean'),
    MAE_thb=('absolute_error','mean'),total_absolute_error=('absolute_error','sum'))
by_band['row_pct']=100*by_band.rows/len(audit)
by_band['error_share_pct']=100*by_band.total_absolute_error/audit.absolute_error.sum()
display(by_band.round(2))
true_positive=audit.true>0
predicted_positive=audit.gate_probability>=best_threshold
confusion=pd.crosstab(true_positive,predicted_positive,
    rownames=['true_nonzero'],colnames=['predicted_nonzero'])
display(confusion)
for label,mask in [('false_zero',true_positive & ~predicted_positive),
                   ('false_nonzero',~true_positive & predicted_positive)]:
    print(label,'rows',int(mask.sum()),'row_pct',round(100*mask.mean(),2),
          'error_share_pct',round(100*audit.loc[mask,'absolute_error'].sum()/audit.absolute_error.sum(),2))

,rows,mean_target,MAE_thb,total_absolute_error,row_pct,error_share_pct
band,,,,,,
zero,40039,0.00,1927.89,7.719067e+07,48.12,32.04
"(0,100]",2006,51.97,50.51,1.013165e+05,2.41,0.04
"(100,1k]",11291,416.98,315.33,3.560373e+06,13.57,1.48
"(1k,10k]",18608,3006.82,584.49,1.087620e+07,22.37,4.51
"(10k,50k]",4470,31148.46,4653.17,2.079968e+07,5.37,8.63
50k+,6786,99423.84,18923.14,1.284124e+08,8.16,53.30


predicted_nonzero,False,True
true_nonzero,,
False,29190,10849
True,12676,30485


false_zero rows 12676 row_pct 15.24 error_share_pct 2.52
false_nonzero rows 10849 row_pct 13.04 error_share_pct 31.96


## 7. Is the positive amount systematically mis-scaled?

The current gate predicts a positive-only MAE estimate above the selected nonzero cutoff. Check a small multiplier on that amount using only paired development-fold predictions. Select the multiplier from four folds and evaluate it on the fifth; keep the original holdout separate. This tests whether the remaining false-positive cost can be reduced without retraining.

In [13]:
scales=np.round(np.arange(.70,1.301,.025),3)
fold_id=paired_gate.fold.to_numpy(dtype=int)
true=paired_gate.true.to_numpy(dtype=float)
probability=paired_gate.probability.to_numpy(dtype=float)
positive_amount=np.maximum(paired_gate.positive_amount.to_numpy(dtype=float),0)
base_pred=best_pair_weight*np.where(probability>=best_threshold,positive_amount,0)+(1-best_pair_weight)*blend_pred
scale_predictions={float(s):best_pair_weight*np.where(probability>=best_threshold,s*positive_amount,0)+(1-best_pair_weight)*blend_pred for s in scales}
outer=[]
for fold in sorted(np.unique(fold_id)):
    fit=fold_id!=fold;check=~fit
    train_scores={s:mean_absolute_error(true[fit],pred[fit]) for s,pred in scale_predictions.items()}
    chosen=min(scales,key=lambda s:(train_scores[float(s)],abs(s-1)))
    outer.append({'fold':fold,'selected_scale':float(chosen),
        'base_MAE_thb':mean_absolute_error(true[check],base_pred[check]),
        'scaled_MAE_thb':mean_absolute_error(true[check],scale_predictions[float(chosen)][check])})
outer=pd.DataFrame(outer)
display(outer.round(2))
print('Mean outer-fold improvement THB:',(outer.base_MAE_thb-outer.scaled_MAE_thb).mean())
print('Improved folds:',int((outer.scaled_MAE_thb<outer.base_MAE_thb).sum()),'of',len(outer))
all_scores={s:mean_absolute_error(true,pred) for s,pred in scale_predictions.items()}
selected_scale=float(min(scales,key=lambda s:(all_scores[float(s)],abs(s-1))))
print('All-development selected scale:',selected_scale,'same-OOF MAE:',all_scores[selected_scale])
print('Original fixed blend OOF MAE:',mean_absolute_error(true,base_pred))
existing_gate=load_saved('t3_positive_gate_cv5_v2')
existing_blend=load_saved('t3_cat_lgb_blend_cv5_v2')
hold_probability=existing_gate.occurrence_model.predict_proba(X_valid)[:,1]
hold_amount=np.maximum(existing_gate.positive_amount_model.predict(X_valid),0)
hold_blend=existing_blend.predict(X_valid)
hold_base=best_pair_weight*np.where(hold_probability>=best_threshold,hold_amount,0)+(1-best_pair_weight)*hold_blend
hold_scaled=best_pair_weight*np.where(hold_probability>=best_threshold,selected_scale*hold_amount,0)+(1-best_pair_weight)*hold_blend
print('Holdout baseline MAE THB:',mean_absolute_error(y_valid,hold_base))
print('Holdout selected-scale MAE THB:',mean_absolute_error(y_valid,hold_scaled))
print('Holdout zero and 50k+ MAE THB:')
for group,mask in [('zero',y_valid==0),('50k_plus',y_valid>=50_000)]:
    print(group,mean_absolute_error(y_valid.loc[mask],hold_base[mask.to_numpy()]),mean_absolute_error(y_valid.loc[mask],hold_scaled[mask.to_numpy()]))

,fold,selected_scale,base_MAE_thb,scaled_MAE_thb
0,1,1.00,2815.58,2815.58
1,2,0.98,2973.69,2976.67
2,3,0.98,2797.07,2801.29
3,4,0.98,2929.51,2938.63
4,5,1.00,2963.76,2963.76


Mean outer-fold improvement THB: -3.2642009440515722
Improved folds: 0 of 5
All-development selected scale: 0.975 same-OOF MAE: 2895.8514592618494
Original fixed blend OOF MAE: 2895.921540742373


Holdout baseline MAE THB: 2845.0992169223155
Holdout selected-scale MAE THB: 2845.2207687432206
Holdout zero and 50k+ MAE THB:
zero 1825.1189842743577 1788.1752198840518
50k_plus 19060.71616308467 19362.852069589502


## 8. Change only the positive amount regressor

The occurrence classifier swap was neutral, and a global amount multiplier failed. Fit a LightGBM MAE model only on positive development rows within each fold. Pair it with the existing fold-local CatBoost occurrence probabilities and the fixed threshold and 80/20 mixture. Compare its amount predictions to the existing positive-only CatBoost on identical folds, including a fixed 50/50 amount blend. The existing holdout remains separate.

In [14]:
amount_key={'dataset':DATA_NAME,'dataset_created_at':card['created_at'],
    'seed':SEED,'folds':5,'model':'LightGBM L1 400x31 positive-only',
    'split':'same stratified development folds as gate'}
amount_parts=[]
for fold,(fit_pos,check_pos) in enumerate(splits,start=1):
    path=RESULTS/f'round4_positive_lgb_fold{fold}_v1.parquet'
    meta=RESULTS/f'round4_positive_lgb_fold{fold}_v1.json'
    if path.exists() and meta.exists():
        assert json.loads(meta.read_text(encoding='utf8'))=={'key':amount_key,'fold':fold}
        part=pd.read_parquet(path)
        assert part.row_pos.tolist()==check_pos.tolist()
        print('Loaded positive LGB fold',fold)
    else:
        assert not (path.exists() or meta.exists()),'Inspect incomplete fold cache'
        fit_y=cv_y.iloc[fit_pos]
        regressor=lgb_regressor()
        regressor.fit(cv_X.iloc[fit_pos].loc[fit_y>0],fit_y.loc[fit_y>0])
        part=pd.DataFrame({'fold':fold,'row_pos':check_pos,
            'true':cv_y.iloc[check_pos].to_numpy(),
            'lgb_positive_amount':regressor.predict(cv_X.iloc[check_pos])})
        part.to_parquet(path,index=False)
        meta.write_text(json.dumps({'key':amount_key,'fold':fold},indent=2)+'\n',encoding='utf8')
        print('Trained positive LGB fold',fold)
    amount_parts.append(part)
amount_oof=pd.concat(amount_parts,ignore_index=True).sort_values('row_pos').reset_index(drop=True)
assert amount_oof.row_pos.equals(paired_gate.row_pos)
assert np.allclose(amount_oof.true,paired_gate.true)
amount_candidates={
    'cat_positive':np.maximum(paired_gate.positive_amount.to_numpy(),0),
    'lgb_positive':np.maximum(amount_oof.lgb_positive_amount.to_numpy(),0),
    'equal_amount_blend':np.maximum(.5*paired_gate.positive_amount.to_numpy()+.5*amount_oof.lgb_positive_amount.to_numpy(),0),
}
amount_rows=[]
for name,amount in amount_candidates.items():
    candidate_gate=np.where(probability>=best_threshold,amount,0)
    candidate_pair=best_pair_weight*candidate_gate+(1-best_pair_weight)*blend_pred
    for fold in sorted(np.unique(fold_id)):
        mask=fold_id==fold
        amount_rows.append({'candidate':name,'fold':fold,
            'gate_MAE_thb':mean_absolute_error(true[mask],candidate_gate[mask]),
            'pair_MAE_thb':mean_absolute_error(true[mask],candidate_pair[mask]),
            'zero_MAE_thb':mean_absolute_error(true[mask & (true==0)],candidate_pair[mask & (true==0)]),
            'tail_MAE_thb':mean_absolute_error(true[mask & (true>=50000)],candidate_pair[mask & (true>=50000)])})
amount_comparison=pd.DataFrame(amount_rows)
display(amount_comparison.pivot(index='fold',columns='candidate',values='pair_MAE_thb').round(2))
display(amount_comparison.groupby('candidate')[['gate_MAE_thb','pair_MAE_thb','zero_MAE_thb','tail_MAE_thb']].mean().round(2))
for name in ('lgb_positive','equal_amount_blend'):
    base=amount_comparison[amount_comparison.candidate=='cat_positive'].set_index('fold').pair_MAE_thb
    challenger=amount_comparison[amount_comparison.candidate==name].set_index('fold').pair_MAE_thb
    print(name,'mean paired gain THB:',(base-challenger).mean(),
          'improved folds:',int((challenger<base).sum()),'of',len(base))

Trained positive LGB fold 1


Trained positive LGB fold 2


Trained positive LGB fold 3


Trained positive LGB fold 4


Trained positive LGB fold 5


candidate,cat_positive,equal_amount_blend,lgb_positive
fold,,,
1,2815.58,2821.53,2861.60
2,2973.69,2970.52,3001.25
3,2797.07,2811.44,2860.40
4,2929.51,2939.14,2978.63
5,2963.76,2969.14,3002.28


,gate_MAE_thb,pair_MAE_thb,zero_MAE_thb,tail_MAE_thb
candidate,,,,
cat_positive,2903.13,2895.92,1927.88,18923.12
equal_amount_blend,2905.51,2902.35,1927.76,19173.04
lgb_positive,2958.67,2940.83,1927.64,19600.94


lgb_positive mean paired gain THB: -44.90880339967471 improved folds: 0 of 5
equal_amount_blend mean paired gain THB: -6.432923470939841 improved folds: 1 of 5
